# 30 Diffbot descarga textos

Al ejecutar la primera celda, se pide la autorización para acceder a Gdrive. A veces falla en el primer intento, volver a intentarlo.



In [41]:
# IMPORTS
import os, re, io, time
import requests
import json
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import gspread
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.http import MediaFileUpload

import pandas as pd

# url_sin_detenidos = 'https://docs.google.com/spreadsheets/d/1xKAdnEC8HP4b5MJ-HrOGKqa7cXE-ue336RsWkIAhnYg/edit?gid=762130750#gid=762130750'

# EDG's copy for testing
url_sin_detenidos = 'https://docs.google.com/spreadsheets/d/1ZNzwwJXkg1nX2UmkXnuwocMBovUuRTc3OT6FxNx0usg/edit?usp=sharing'


try:
    from google.colab import auth, userdata
    import google.auth

    # Running in Colab
    auth.authenticate_user()
    creds, _ = google.auth.default()
    client = gspread.authorize(creds)

    api_key = userdata.get('DIFFBOT_API_KEY')

except ImportError:
    from google.oauth2.credentials import Credentials

    # Running locally or outside Colab
    SCOPES = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ]

    oauth_client_path = Path.cwd() / ".." / "secrets" / "oauth_client.json"
    token_path = Path.cwd() / ".." / "secrets" / "token.json"

    # Reuse a cached token across runs; only pops up the browser login when needed
    creds = None
    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(str(oauth_client_path), SCOPES)
            creds = flow.run_local_server(port=0)
        token_path.write_text(creds.to_json())

    # creds = Credentials.from_service_account_file(
    #     "../secrets/credentials.json", scopes=SCOPES
    # )
    client = gspread.authorize(creds)

    load_dotenv()  # Automatically finds .env file
    api_key = os.getenv('DIFFBOT_API_KEY')


spreadsheet = client.open_by_url(url_sin_detenidos)
drive_service = build('drive', 'v3', credentials=creds)

# Folder with articles
url_folder = "https://drive.google.com/drive/folders/1i5Jl-Oho9k6PJPk4apsaUTwZbTCh8-oV?usp=sharing"
folder_id = re.search(r"/folders/([^/?]+)", url_folder).group(1)



### Dataframe con artículos (sin el texto) registrados por el RSS

In [42]:
# ws = spreadsheet.get_worksheet(0)
ws = spreadsheet.worksheet('INBOX')

# Get raw cell values as a 2D list
raw_values = ws.get_all_values()
# Add a column for 'Archivo' if it doesn't exist
if not 'Archivo' in raw_values[0]:
    # Find the column index of 'Puntaje' in the first row (1-based index)
    header_row = ws.row_values(1)
    puntaje_col_index = header_row.index('Puntaje') + 1

    # Insert a new empty column right after 'Puntaje', shifting the rest to the right
    ws.insert_cols([[]], puntaje_col_index + 1)

    # Set the header of the newly inserted column
    ws.update_cell(1, puntaje_col_index + 1, 'Archivo')

# Define your own custom header names
headers = ['Fecha_deteccion', 'Medio', 'Titulo', 'Link', 'Keyword_detectada',
        'Fuente', 'Revisado', 'Validado', 'Observaciones', 'Estado_IA',
        'Palabras_detectadas', 'Puntaje', 'archivo', 'Puntaje_solo_titulo', 'otro_2']

# Map values to records (skipping row 0 if row 0 has the old headers)
all_records = gspread.utils.to_records(headers, raw_values[1:])

df_sd = pd.DataFrame(all_records)

# Add column with the actual row number in the Google Spreadsheet
# (raw_values[1:] starts at spreadsheet row 2)
df_sd["gdrive_index"] = range(2, 2 + len(df_sd))

# Remove invalid rows: no Title
# Filter out empty/whitespace strings and NaNs
col = df_sd.columns[2]
df_sd = df_sd[df_sd[col].astype(str).str.strip().ne("") & df_sd[col].notna()]

# Reset index, keep the old index for reference, matching the row number in the gdrive sheet.
# df_sd.reset_index(names="gdrive_index", inplace=True)
df_sd.reset_index(drop=True, inplace=True)

# Remove column "Medio", it's always equal to "Google News"
df_sd.drop(columns=["Medio"], inplace=True)

# Remove column "Fuente", it shows the search source URL, not the url for the article
df_sd.drop(columns=["Fuente"], inplace=True)

# Últimos artículos
df_sd.iloc[-5:]

,Fecha_deteccion,Titulo,Link,Keyword_detectada,Revisado,Validado,Observaciones,Estado_IA,Palabras_detectadas,Puntaje,archivo,Puntaje_solo_titulo,otro_2,gdrive_index
402,10/9/2026,Video: un policía de la Ciudad mató a tiros a ...,https://news.google.com/rss/articles/CBMi8AFBV...,POLICIA + VIOLENCIA GENERAL,FALSE,PENDIENTE,,,POLICIA: policia | VIOLENCIA_GENERAL: mato,,,,,407
403,10/9/2026,Berisso: importante movilización contra la rep...,https://news.google.com/rss/articles/CBMiyAFBV...,POLICIA + POSIBLE VICTIMA + VIOLENCIA POLICIAL,FALSE,PENDIENTE,,,"POLICIA: policia, policial | POSIBLE_VICTIMA: ...",,,,,408
404,11/9/2026,El riesgo de los policías sin uniforme en las ...,https://news.google.com/rss/articles/CBMirwFBV...,POLICIA + POSIBLE VICTIMA,FALSE,PENDIENTE,,,POLICIA: policia | POSIBLE_VICTIMA: protesta,,,,,409
405,11/9/2026,Incidentes frente al Congreso: 26 detenidos y ...,https://news.google.com/rss/articles/CBMitwFBV...,POLICIA + VICTIMA + VIOLENCIA GENERAL + CABA,FALSE,PENDIENTE,,,POLICIA: policia | VICTIMA: detenido | VIOLENC...,,,,,410
406,12/9/2026,Citan a indagatoria al policía de la Ciudad qu...,https://news.google.com/rss/articles/CBMizwFBV...,POLICIA + VIOLENCIA GENERAL + CABA,FALSE,PENDIENTE,,,"POLICIA: policia | VIOLENCIA_GENERAL: asesino,...",,,,,411


In [ ]:
# Link of the nth article in the DataFrame
print(df_sd.iloc[402]['Link'])

In [44]:
df_sd.iloc[390:401]


,Fecha_deteccion,Titulo,Link,Keyword_detectada,Revisado,Validado,Observaciones,Estado_IA,Palabras_detectadas,Puntaje,archivo,Puntaje_solo_titulo,otro_2,gdrive_index
390,6/9/2026,Megaoperativo con 1.100 policías en los grande...,https://news.google.com/rss/articles/CBMiwgFBV...,POLICIA + VIOLENCIA POLICIAL,FALSE,PENDIENTE,,,POLICIA: policia | VIOLENCIA_POLICIAL: megaope...,,,,,395
391,6/9/2026,Presentación de la Guía de Prevención sobre Ab...,https://news.google.com/rss/articles/CBMizgFBV...,POLICIA + VIOLENCIA GENERAL,FALSE,PENDIENTE,,,POLICIA: policia | VIOLENCIA_GENERAL: abuso,,,,,396
392,6/9/2026,DISTINCIÓN A LA SUPERINTENDENCIA DE VIOLENCIA ...,https://news.google.com/rss/articles/CBMi5AFBV...,POLICIA + VIOLENCIA GENERAL,FALSE,PENDIENTE,,,POLICIA: policia | VIOLENCIA_GENERAL: violencia,,,,,397
393,6/9/2026,Megaoperativo policial de la Ciudad: más de 1....,https://news.google.com/rss/articles/CBMiwAFBV...,POLICIA + VIOLENCIA POLICIAL,FALSE,PENDIENTE,,,"POLICIA: policia, efectivo, policial | VIOLENC...",,20260913_00005.txt,,,398
394,7/9/2026,Un efectivo de la Federal asesinó a un joven d...,https://news.google.com/rss/articles/CBMitAFBV...,POLICIA + VIOLENCIA GENERAL,FALSE,PENDIENTE,,,POLICIA: efectivo | VIOLENCIA_GENERAL: asesino,,20260913_00006.txt,,,399
395,7/9/2026,"""La patria no se vende"": Protesta contra proye...",https://news.google.com/rss/articles/CBMi8AFBV...,POLICIA + POSIBLE VICTIMA + VIOLENCIA POLICIAL,FALSE,PENDIENTE,,,"POLICIA: policia, policial | POSIBLE_VICTIMA: ...",,20260913_00007.txt,,,400
396,7/9/2026,Un policía mató a un delincuente menor de edad...,https://news.google.com/rss/articles/CBMi0wFBV...,POLICIA + VIOLENCIA GENERAL,FALSE,PENDIENTE,,,POLICIA: policia | VIOLENCIA_GENERAL: mato,,20260913_00008.txt,,,401
397,8/9/2026,Caballito: la Policía reprimió a personas en s...,https://news.google.com/rss/articles/CBMivgFBV...,POLICIA + VICTIMA + VIOLENCIA POLICIAL + CABA,FALSE,PENDIENTE,,,POLICIA: policia | VICTIMA: situacion de calle...,,20260913_00009.txt,,,402
398,8/9/2026,Banderazo violento en el Obelisco: La Policía ...,https://news.google.com/rss/articles/CBMi4wFBV...,POLICIA + VIOLENCIA GENERAL + CABA,FALSE,PENDIENTE,,,POLICIA: policia | VIOLENCIA_GENERAL: violento...,,20260913_00010.txt,,,403
399,8/9/2026,Policía concretó el traslado desde CABA de dos...,https://news.google.com/rss/articles/CBMizwFBV...,POLICIA + VICTIMA + CABA,FALSE,PENDIENTE,,,POLICIA: policia | VICTIMA: detenido | CABA: caba,,,,,404


## Diffbot

In [21]:
def inicializar_contador_archivos(df=None, columna="archivo"):
    """
    Initializes the global file counter based on existing filenames
    in `df[columna]`, so new files continue the numbering instead of
    restarting at 0. Only considers filenames matching today's date.
    """
    global _file_counter
    _file_counter = 0

    if df is None or columna not in df.columns:
        return

    fecha = datetime.now().strftime("%Y%m%d")
    pattern = re.compile(rf"^{fecha}_(\d{{5}})\.txt$")

    max_counter = 0
    for value in df[columna].dropna():
        match = pattern.match(str(value).strip())
        if match:
            max_counter = max(max_counter, int(match.group(1)))

    _file_counter = max_counter

def generar_nombre_archivo():
    """
    Generates a filename like YYYYMMDD_NNNNNN.txt using today's date
    and an incrementing counter (per run).
    """
    global _file_counter
    _file_counter += 1
    fecha = datetime.now().strftime("%Y%m%d")
    return f"{fecha}_{_file_counter:05d}.txt"

def guardar_texto_en_drive(texto, nombre_archivo=None, folder_id=folder_id):
    """
    Saves `texto` as a .txt file inside the Google Drive folder `folder_id`.
    Returns the created file's id and name.
    """
    if nombre_archivo is None:
        nombre_archivo = generar_nombre_archivo()

    file_metadata = {
        "name": nombre_archivo,
        "parents": [folder_id],
    }
    media = MediaIoBaseUpload(
        io.BytesIO((texto or "").encode("utf-8")),
        mimetype="text/plain",
        resumable=False,
    )
    created_file = drive_service.files().create(
        body=file_metadata, media_body=media, fields="id, name"
    ).execute()

    return created_file


In [47]:
# articles_dir = Path.cwd() / ".." / "data" / "articles"


# df = pd.read_csv(csv_path)
# print(f"Number of rows in DataFrame: {len(df)}")

# Rows to process from the DataFrame
# If not set, it will fetch all rows that do not have a diffbot response already.
n_i = 392
n_f = 401

if n_i >= n_f:
    raise ValueError("Starting index n_i must be less than ending index n_f.")

print(f"Processing rows {n_i} to {n_f} from DataFrame...")
print("Waiting 1 minute between requests to avoid hitting the rate limit...")

# We are not saving the full diffbot response. In the future, this
# could be modified to store more detailed information from the response.
# Perhaps, saving the full response to a separate file could be considered.
# if "diffbot_response" not in df_sd.columns:
#     df_sd["diffbot_response"] = pd.NA

# Index of column for saving filenames with articles' texts
header_row = ws.row_values(1)
archivo_col_index = header_row.index('Archivo') + 1

url = f"https://api.diffbot.com/v3/article?token={api_key}"
headers = {}

subset = df_sd.iloc[n_i:n_f]

# Filter only rows that do not have a diffbot response already.
def needs_diffbot_response(value):
    if pd.isna(value):
        return True
    if not isinstance(value, str):
        return True
    text = value.strip()
    if not text:
        return True
    # try:
    #     parsed = json.loads(text)
    # except (ValueError, TypeError):
    #     return True
    return False
    # return not (isinstance(parsed, dict) and "request" in parsed)

missing_response = subset["archivo"].apply(needs_diffbot_response)

inicializar_contador_archivos(df_sd, columna="archivo")

for index, row in subset[missing_response].iterrows():
    print(f"Processing row {index} (ID = {row.get('ID')})...", end="")
    link = row.get("Link")
    if pd.isna(link) or not str(link).strip():
        print(f" skipping, missing link")
        continue

    try:
        params = {
            "url": link
        }
        response = requests.request("GET", url, params=params, headers=headers)
        print(f" status={response.status_code}")
        response_json = response.json()
        obj = json.loads(response.text)

        json_obj = (obj.get('objects') or [{}])[0]
        article_text = json_obj.get('text')
        filename = generar_nombre_archivo()
        created_file = guardar_texto_en_drive(article_text, nombre_archivo=filename, folder_id=folder_id)
        ws.update_cell(row.gdrive_index, archivo_col_index, filename)

    except requests.RequestException as exc:
        print(f" request failed: {exc}")

    time.sleep(65)  # Sleep for 65 seconds to avoid hitting the rate limit


Processing rows 392 to 401 from DataFrame...
Waiting 1 minute between requests to avoid hitting the rate limit...
Processing row 392 (ID = None)... status=200
Processing row 399 (ID = None)... status=200
Processing row 400 (ID = None)... status=200


## Consultar créditos en Diffbot

Se pueden descargar hasta 2000 textos por mes.

In [48]:
url_credits = f"https://api.diffbot.com/v4/account?token={api_key}"

headers = {"accept": "application/json"}

response = requests.get(url_credits, headers=headers)

response_json = response.json()
total_credits = 0
for u in response_json['usage']:
    total_credits += u['credits']
print("-")
print(f"Total créditos usados en los últimos 31 días: {total_credits}")
print("-")

-
Total créditos usados en los últimos 31 días: 28
-


### Comentarios

Running local:

If 'token.json' does not exist, it should pop up a Google login/consent window the first time since it's a fresh OAuth client.

Create an OAuth Client ID (not a service account) in Google Cloud Console → APIs & Services → Credentials → "Create Credentials" → "OAuth client ID" → Application type: Desktop app. Download it as e.g. secrets/oauth_client.json.

Consent screen needs this:
- Go to the Google Cloud Console.
- Select your project (Mapa).
- In the left menu, go to APIs & Services → OAuth consent screen (Pantalla de consentimiento de OAuth).
- Scroll down to the Test users (Usuarios de prueba) section. (Audience)
- Click + ADD USERS and enter your personal Gmail address.
- Click Save. (Check that the user was added to the list, this last save is tricky)